In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import pandas as pd

BASE_DIR = "drive/MyDrive/CSML/data"
SWEEP_DIR = os.path.join(BASE_DIR, "imbalance_sweep_results")

summary = pd.read_csv(os.path.join(SWEEP_DIR, "imbalance_sweep_summary.csv"))
with open(os.path.join(SWEEP_DIR, "imbalance_sweep_curves.json")) as f:
    curves = json.load(f)

# ----------------------------------------------------------------
# Step 1: get each source's VAL set majority class (fixed, since
# only the train split was resampled -- val is untouched).
# ----------------------------------------------------------------
val_majority = {}
val_real_frac = {}
for source in summary["source"].unique():
    df = pd.read_csv(os.path.join(BASE_DIR, "clean", f"{source}_clean.csv"))
    val_df = df[df["split"] == "val"]
    real_frac = (val_df["label"] == 1).mean()
    val_real_frac[source] = real_frac
    val_majority[source] = 1 if real_frac >= 0.5 else 0
    print(f"{source}: val set real fraction = {real_frac:.4f} (majority class = {val_majority[source]})")

# ----------------------------------------------------------------
# Step 2: for every COLLAPSED run, check:
#   (a) does the collapse direction match the val set's own majority class?
#   (b) was the "best" checkpoint selected at/near the collapse epoch --
#       i.e. did an early degenerate solution get locked in as "best"
#       before real learning had a chance to happen?
# ----------------------------------------------------------------
rows = []
for _, run in summary.iterrows():
    if not run["collapsed_final"]:
        continue

    source = run["source"]
    collapse_dir = 1 if run["test_pred_real_frac"] >= 0.5 else 0
    matches_val_majority = (collapse_dir == val_majority[source])

    run_id = (
        f"{source}_ratio{run['target_real_ratio']}_seed{run['seed']}_"
        f"{'weighted' if run['class_weighted'] else 'unweighted'}"
    )
    curve = curves.get(run_id)
    if curve is None:
        print(f"[warning] no curve found for {run_id}")
        continue

    val_f1_curve = curve["val_f1_curve"]
    best_epoch = int(np.argmax(val_f1_curve))
    first_collapse_epoch = run["epoch_first_collapse"]

    # "locked in at collapse" = the best-F1 checkpoint was chosen at or within
    # 1 epoch of the first detected collapse (i.e. a degenerate solution won
    # the val-F1 race before real learning happened)
    locked_in_at_collapse = (
        first_collapse_epoch >= 0 and abs(best_epoch - first_collapse_epoch) <= 1
    )

    rows.append({
        "run_id": run_id,
        "source": source,
        "target_real_ratio": run["target_real_ratio"],
        "seed": run["seed"],
        "class_weighted": run["class_weighted"],
        "collapse_direction": "real" if collapse_dir == 1 else "fake",
        "val_set_majority": "real" if val_majority[source] == 1 else "fake",
        "matches_val_majority": matches_val_majority,
        "epoch_first_collapse": first_collapse_epoch,
        "best_checkpoint_epoch": best_epoch,
        "locked_in_at_collapse": locked_in_at_collapse,
        "best_val_macro_f1": run["best_val_macro_f1"],
    })

analysis_df = pd.DataFrame(rows)
out_path = os.path.join(SWEEP_DIR, "collapse_direction_analysis.csv")
analysis_df.to_csv(out_path, index=False)

print(f"\n=== Collapse direction analysis ({len(analysis_df)} collapsed runs) ===")
print(analysis_df.to_string(index=False))

print(f"\n=== Summary ===")
print(f"Runs matching val-set majority class:     {analysis_df['matches_val_majority'].sum()} / {len(analysis_df)}")
print(f"Runs where best checkpoint was locked in\n"
      f"  at/near the first collapse epoch:        {analysis_df['locked_in_at_collapse'].sum()} / {len(analysis_df)}")

print(f"\nSaved to: {out_path}")
